In [42]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout


# Para la optimización de hiperparámetros con LSTM usaremos Keras Tuner:
import keras_tuner as kt

In [43]:
file_path = r"datos_preprocesados_winsorizado.csv"

df = pd.read_csv(file_path)  

In [44]:
df['casa_id'] = df['casa_id'].astype(str)

In [45]:
df_onehot = pd.get_dummies(df['casa_id'], prefix='casa')
df = pd.concat([df, df_onehot], axis=1)

In [46]:

# Crear lag features por cada casa
df['lag_1'] = df.groupby('casa_id')['consumo_kwh'].shift(1)   # consumo 1 hora anterior
df['lag_24'] = df.groupby('casa_id')['consumo_kwh'].shift(24)  # consumo a la misma hora del día anterior

# Eliminamos filas sin lag (al comienzo de la serie para cada casa)
df = df.dropna(subset=['lag_1', 'lag_24'])

In [47]:
features = [
    'consumo_kwh',        # Variable original (también target)
    'coste_euros',        # Si consideras que aporta información adicional
    'dayofweek',
    'lag_1', 'lag_24'     # Si ya las tienes creadas
] + [col for col in df.columns if col.startswith('casa_')]


In [48]:
# Ordenamos por 'casa_id' y 'timestamp' para crear secuencias sin mezclar casas
df.sort_values(['casa_id', 'timestamp'], inplace=True)
print("Datos cargados y ordenados:" , df.head())

Datos cargados y ordenados:       casa_id            timestamp  consumo_kwh  coste_euros  year  month  \
72384   10296  2018-02-09 01:00:00       0.6135     0.162221  2018      2   
72385   10296  2018-02-09 02:00:00       0.6135     0.130089  2018      2   
72386   10296  2018-02-09 03:00:00       0.6135     0.134510  2018      2   
72387   10296  2018-02-09 04:00:00       0.6135     0.130873  2018      2   
72388   10296  2018-02-09 05:00:00       0.6135     0.148405  2018      2   

       day  hour  dayofweek  is_weekend  ...  casa_8162  casa_8284  casa_8289  \
72384    9     1          4           0  ...      False      False      False   
72385    9     2          4           0  ...      False      False      False   
72386    9     3          4           0  ...      False      False      False   
72387    9     4          4           0  ...      False      False      False   
72388    9     5          4           0  ...      False      False      False   

       casa_9417  casa

In [ ]:
# # Verificar la primera y la última fecha en el DataFrame
# primera_fecha = df['timestamp'].min()
# ultima_fecha = df['timestamp'].max()

# print(f"Primera fecha en el DataFrame: {primera_fecha}")
# print(f"Última fecha en el DataFrame: {ultima_fecha}")

Primera fecha en el DataFrame: 2017-05-31 01:00:00
Última fecha en el DataFrame: 2021-05-31 00:00:00


In [58]:
window_size = 24  # Tamaño de la ventana (24 horas)

In [55]:
class DataGenerator(tf.keras.utils.Sequence):
    def __init__(self, df, features, window_size, batch_size=32, scaler=None, indices=None):
        """
        df: DataFrame preprocesado, ordenado por 'casa_id' y 'timestamp'.
        features: lista de columnas a utilizar.
        window_size: longitud de la secuencia (ej. 24 horas).
        batch_size: tamaño del batch.
        scaler: opcional, escalador para normalizar las variables continuas.
        indices: lista de índices (tuplas (casa, start_idx)) a usar. Si None, se usan todas.
        """
        self.df = df
        self.features = features
        self.window_size = window_size
        self.batch_size = batch_size
        self.scaler = scaler
        self.groups = {}
        self.all_indices = []
        # Para cada casa, obtenemos sus datos ordenados y calculamos los índices válidos para secuencias.
        for casa, group in df.groupby('casa_id'):
            group = group.sort_values('timestamp')
            group_values = group[features].values.astype(np.float32)
            self.groups[casa] = group_values
            n = len(group_values)
            if n > window_size:
                for i in range(n - window_size):
                    self.all_indices.append((casa, i))
        # Si se pasan índices específicos, los usamos; de lo contrario, usamos todos.
        if indices is not None:
            self.indices = indices
        else:
            self.indices = self.all_indices
    
    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))
    
    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size : (idx + 1) * self.batch_size]
        X_batch = []
        y_batch = []
        for casa, start_idx in batch_indices:
            group_values = self.groups[casa]
            # Secuencia de entrada: ventana de 'window_size' horas
            X_seq = group_values[start_idx : start_idx + self.window_size]
            # Target: consumo_kwh de la hora siguiente (columna 0)
            y_val = group_values[start_idx + self.window_size][0]
            # Si se proporciona un escalador, transformamos la secuencia
            if self.scaler is not None:
                X_seq = self.scaler.transform(X_seq)
            X_batch.append(X_seq)
            y_batch.append(y_val)
        return np.array(X_batch, dtype=np.float32), np.array(y_batch, dtype=np.float32)


In [56]:
scaler = None

# Creamos el generador completo con todas las secuencias.
generator_full = DataGenerator(df, features, window_size, batch_size=32, scaler=scaler)

In [57]:
# ### 3. División de Datos en Entrenamiento y Validación
# Dividimos la lista de todos los índices en 80% para entrenamiento y 20% para validación.
all_indices = generator_full.all_indices
split_idx = int(0.8 * len(all_indices))
train_indices = all_indices[:split_idx]
val_indices = all_indices[split_idx:]

train_gen = DataGenerator(df, features, window_size, batch_size=32, scaler=scaler, indices=train_indices)
val_gen = DataGenerator(df, features, window_size, batch_size=32, scaler=scaler, indices=val_indices)

print("Número de batches en entrenamiento:", len(train_gen))
print("Número de batches en validación:", len(val_gen))

Número de batches en entrenamiento: 66588
Número de batches en validación: 16647


In [59]:
# ### 4. Definir el Modelo LSTM y la Función para Keras Tuner
def build_model(hp):
    model = Sequential()
    model.add(LSTM(units=hp.Choice('units', values=[32, 64, 128]),
                   activation='tanh',
                   return_sequences=hp.Boolean('return_sequences', default=False),
                   input_shape=(window_size, len(features))))
    if hp.Boolean('dropout'):
        model.add(Dropout(rate=hp.Float('dropout_rate', min_value=0.1, max_value=0.5, step=0.1)))
    if hp.Boolean('second_layer'):
        model.add(LSTM(units=hp.Choice('units2', values=[32, 64, 128]), activation='tanh'))
        if hp.Boolean('dropout2'):
            model.add(Dropout(rate=hp.Float('dropout_rate2', min_value=0.1, max_value=0.5, step=0.1)))
    model.add(Dense(1))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=hp.Choice('learning_rate', values=[1e-3, 1e-4])),
                  loss='mse', metrics=['mae'])
    return model

In [60]:
# ### 5. Optimización de Hiperparámetros con Keras Tuner
tuner = kt.RandomSearch(build_model,
                        objective='val_loss',
                        max_trials=10,
                        executions_per_trial=1,
                        directory='kt_dir',
                        project_name='global_lstm_generator')

tuner.search(train_gen, epochs=20, validation_data=val_gen,
             callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)],
             verbose=1)

best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()

Trial 4 Complete [01h 46m 33s]
val_loss: 0.012876446358859539

Best val_loss So Far: 0.012876446358859539
Total elapsed time: 03h 00m 27s

Search: Running Trial #5

Value             |Best Value So Far |Hyperparameter
128               |32                |units
False             |True              |return_sequences
False             |True              |dropout
False             |True              |second_layer
0.0001            |0.0001            |learning_rate
128               |64                |units2
0.1               |0.1               |dropout_rate
False             |False             |dropout2

Epoch 1/20
66588/66588 ━━━━━━━━━━━━━━━━━━━━ 616s 9ms/step - loss: 0.0375 - mae: 0.1557 - val_loss: 0.0171 - val_mae: 0.0926
Epoch 2/20
66588/66588 ━━━━━━━━━━━━━━━━━━━━ 586s 9ms/step - loss: 0.0204 - mae: 0.1023 - val_loss: 0.0166 - val_mae: 0.0877
Epoch 3/20
66588/66588 ━━━━━━━━━━━━━━━━━━━━ 591s 9ms/step - loss: 0.0188 - mae: 0.0958 - val_loss: 0.0161 - val_mae: 0.0932
Epoch 4/20
66588/6

KeyboardInterrupt: 

In [ ]:
# ### 6. Evaluación del Modelo
# Evaluamos el modelo en el conjunto de validación y graficamos algunas predicciones.
loss, mae = best_model.evaluate(val_gen)
print("Validation Loss:", loss, "Validation MAE:", mae)

# Realizamos predicciones en el generador de validación.
# Para graficar, extraemos el primer batch del generador de validación.
X_val, y_val = val_gen[0]
y_val_pred = best_model.predict(X_val).flatten()

plt.figure(figsize=(12,6))
plt.plot(y_val, label='Real')
plt.plot(y_val_pred, label='Predicción')
plt.legend()
plt.title('Predicción Global LSTM con Data Generator (primer batch de validación)')
plt.xlabel('Muestras del batch')
plt.ylabel('Consumo (kWh)')
plt.show()